# Chain-of-Thought Prompting

Chain-of-thought (CoT) prompting elicits intermediate reasoning steps before a final answer. It is one of the highest-leverage prompting techniques for arithmetic, commonsense, and symbolic reasoning tasks — often matching fine-tuned model performance with zero weight updates.

## The Discovery

Wei et al. (2022) showed that including reasoning steps in few-shot examples dramatically improved performance on arithmetic and reasoning benchmarks. Before this work, scaling laws suggested you needed more parameters to improve reasoning. CoT showed that prompting structure could substitute for scale.

The key experiments:
- **GSM8K**: models prompted with step-by-step solutions scored 40-50% higher than same-size models with direct answer prompting
- **MGSM**: multilingual math — CoT improvements transferred across languages
- **StrategyQA**: commonsense reasoning — CoT enabled models to break down multi-hop questions

**Zero-shot CoT** (Kojima et al. 2022): simply appending 'Let's think step by step' to the prompt produces significant CoT effects without any few-shot examples. This phrase became one of the most studied prompts in LLM research.

## Why CoT Works

Several complementary explanations:

**Decomposition**: breaking a problem into sub-steps reduces the difficulty of each individual step. A model that cannot compute 23 × 47 in one step can compute 23 × 40, then 23 × 7, then add them.

**Intermediate token allocation**: the autoregressive generation process allocates a fixed compute budget per token. CoT trades tokens for effective compute — more tokens = more forward passes = more computation applied to the problem.

**Implicit program execution**: reasoning chains resemble program execution traces. The model has seen traces of arithmetic and logical reasoning in pretraining, so this format activates the right internal representations.

**Faithfulness question**: CoT is not always a faithful explanation of the model's internal computation. The model may arrive at the correct answer via other internal mechanisms and generate a post-hoc plausible-sounding chain. This is an active research area.

In [ ]:
from dataclasses import dataclass
from typing import Callable
import re

@dataclass
class CoTResult:
    problem: str
    chain: str
    answer: str
    correct: bool
    strategy: str

def build_cot_prompt(problem: str, examples: list, zero_shot: bool = False) -> str:
    if zero_shot:
        return f'{problem}\nLet\'s think step by step.'
    parts = []
    for ex in examples:
        parts.append(f'Q: {ex["problem"]}')
        parts.append(f'A: {ex["chain"]}')
        parts.append(f'Answer: {ex["answer"]}\n')
    parts.append(f'Q: {problem}')
    parts.append('A:')
    return '\n'.join(parts)

def extract_answer(chain: str) -> str:
    # Look for 'answer is X' or 'the answer: X' or final number
    match = re.search(r'(?:answer(?:\s+is)?|therefore)[:\s]+([\d,\.]+)', chain, re.I)
    if match:
        return match.group(1).replace(',', '')
    numbers = re.findall(r'[\d,]+(?:\.\d+)?', chain)
    return numbers[-1].replace(',','') if numbers else ''

def run_cot(problems: list, model_fn: Callable, examples: list = None,
            zero_shot: bool = False) -> list:
    results = []
    for prob in problems:
        prompt = build_cot_prompt(prob['problem'], examples or [], zero_shot)
        chain = model_fn(prompt)
        answer = extract_answer(chain)
        correct = str(prob.get('answer', '')).strip() in chain or answer == str(prob.get('answer',''))
        results.append(CoTResult(
            problem=prob['problem'], chain=chain, answer=answer,
            correct=correct, strategy='zero_shot' if zero_shot else 'few_shot'
        ))
    return results

few_shot_examples = [
    {
        'problem': 'A store has 15 apples. They sell 7 and receive 12 more. How many apples?',
        'chain': 'Start with 15. After selling 7: 15 - 7 = 8. After receiving 12: 8 + 12 = 20.',
        'answer': '20'
    },
    {
        'problem': 'If 3 workers take 6 days to build a wall, how many days for 9 workers?',
        'chain': 'Total worker-days = 3 * 6 = 18. With 9 workers: 18 / 9 = 2 days.',
        'answer': '2'
    },
]

test_problems = [
    {'problem': 'A farmer has 120 eggs. He sells 45 and packs the rest in boxes of 15. How many boxes?', 'answer': '5'},
    {'problem': 'Train A travels 60 mph. Train B travels 80 mph. They start 280 miles apart moving toward each other. When do they meet?', 'answer': '2'},
]

def mock_cot_model(prompt):
    if 'eggs' in prompt:
        return 'Start with 120 eggs. After selling 45: 120 - 45 = 75 eggs remain. Boxes needed: 75 / 15 = 5 boxes. The answer is 5.'
    if 'Train' in prompt:
        return 'Combined speed = 60 + 80 = 140 mph. Distance = 280 miles. Time = 280 / 140 = 2 hours. The answer is 2.'
    return 'I need to work through this carefully. The answer is 42.'

results = run_cot(test_problems, mock_cot_model, few_shot_examples)
for r in results:
    print(f'Problem: {r.problem}')
    print(f'Chain: {r.chain}')
    print(f'Answer: {r.answer} | Correct: {r.correct}\n')

## Faithfulness Evaluation

CoT chains are not always faithful representations of how the model reached its answer. Tests for faithfulness:

**Counterfactual intervention**: modify an intermediate step to be wrong and check if the final answer changes correspondingly. If the answer stays correct despite a broken chain, the chain is not causally driving the answer.

**Truncation test**: remove the chain and give only the answer — if accuracy stays similar, the chain was not necessary.

**Probing**: check if intermediate claims in the chain are actually represented in the model's internal activations at that point in generation.

Faithfulness varies significantly by task. For arithmetic, chains tend to be more faithful (each step is a discrete computation). For commonsense reasoning, chains are often post-hoc rationalizations.

In [ ]:
def faithfulness_eval(problems: list, model_fn: Callable, corrupt_fn: Callable) -> dict:
    correct_with_chain = 0
    correct_after_corruption = 0
    n = len(problems)
    for prob in problems:
        # Original chain
        chain = model_fn(prob['problem'])
        answer = extract_answer(chain)
        original_correct = answer == str(prob.get('answer', ''))
        if original_correct:
            correct_with_chain += 1
        # Corrupted chain: inject a wrong intermediate step
        corrupted = corrupt_fn(chain)
        corrupted_answer = extract_answer(corrupted)
        # If answer changes after corruption, chain was causally driving it
        if original_correct and corrupted_answer != answer:
            correct_after_corruption += 1
    # High faithfulness: answer changes when chain is corrupted
    faithfulness = correct_after_corruption / max(1, correct_with_chain)
    return {
        'n': n,
        'accuracy': correct_with_chain / n,
        'faithfulness_rate': faithfulness,
        'interpretation': 'faithful' if faithfulness > 0.6 else 'possibly post-hoc'
    }

def corrupt_chain(chain: str) -> str:
    # Replace the first number in the chain with a wrong value
    return re.sub(r'\b(\d+)\b', lambda m: str(int(m.group(1)) + 99), chain, count=1)

faith = faithfulness_eval(test_problems, mock_cot_model, corrupt_chain)
print('Faithfulness evaluation:')
for k, v in faith.items():
    print(f'  {k}: {v}')

## Zero-Shot vs Few-Shot CoT

When to use each:

**Zero-shot CoT** ('Let's think step by step'): when you don't have labeled examples, when the task is novel, or when you want to avoid few-shot contamination in evaluation. Slightly lower accuracy but much simpler prompts.

**Few-shot CoT**: when you have a few high-quality solved examples, when the task has a specific reasoning format you want to enforce, or when zero-shot doesn't reach acceptable accuracy. The examples teach the model not just the format but the strategy for your specific task.

**Auto-CoT** (Zhang et al. 2022): automatically generate diverse few-shot examples by clustering questions and using zero-shot CoT to generate chains for each cluster center. Avoids hand-crafting examples while getting few-shot performance.